# 🤟 Wasel v4 Pro — Live Sign Language Translator
### PSL Kaggle Dataset + Live Webcam Translation

**Pipeline:**
1. Cell 1: Install dependencies
2. Cell 2: Download PSL dataset & train classifier
3. Cell 3: Setup live engine
4. Cell 4: Launch live translator

> ⚡ **Trained on real PSL data — translates in real-time!**

In [ ]:
#@title 📦 Cell 1: Install Dependencies

print("⏳ Installing dependencies...")

!pip install -q \
    "gradio>=5.0.0" \
    "mediapipe>=0.10.0" \
    "scikit-learn>=1.3.0" \
    "opencv-python-headless" \
    "kaggle" \
    "kagglehub"

print("\n🔍 Verifying...")
import importlib
for name, mod in {"gradio": "gradio", "mediapipe": "mediapipe", "sklearn": "sklearn", "cv2": "cv2"}.items():
    try:
        m = importlib.import_module(mod)
        print(f"   ✅ {name}: {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"   ❌ {name}: {e}")

print("\n🎉 Ready! Proceed to Cell 2.")

In [ ]:
#@title 🗄️ Cell 2: Download PSL Dataset & Train Classifier
#@markdown Paste your Kaggle API Token (starts with KGAT_)
#@markdown Get it from: https://www.kaggle.com/settings → API

KAGGLE_API_TOKEN = "" #@param {type:"string"}

import os, json, sqlite3, pickle, glob
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ─── Auth: New KGAT_ token format ───
if not KAGGLE_API_TOKEN:
    raise ValueError("❌ Paste your Kaggle API Token (KGAT_...) above!")

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN.strip()
print("   ✅ Kaggle API Token set.")

# ─── Download via kagglehub (supports new token format) ───
DATASET_DIR = "/content/psl_dataset"
DB_PATH = None
download_success = False

if not os.path.exists(DATASET_DIR) or len(os.listdir(DATASET_DIR)) == 0:
    os.makedirs(DATASET_DIR, exist_ok=True)
    print("⏳ Downloading PSL dataset...")

    # Method 1: kagglehub (supports KGAT_ tokens)
    try:
        import kagglehub
        path = kagglehub.dataset_download("saadbutt32/pakistan-sign-language-dataset")
        print(f"   ✅ Downloaded to: {path}")
        DATASET_DIR = path
        download_success = True
    except Exception as e1:
        print(f"   ⚠️ kagglehub failed: {e1}")

    # Method 2: kaggle CLI
    if not download_success:
        try:
            !KAGGLE_API_TOKEN={KAGGLE_API_TOKEN} kaggle datasets download -d saadbutt32/pakistan-sign-language-dataset -p {DATASET_DIR} --unzip -q
            if os.listdir(DATASET_DIR):
                download_success = True
                print("   ✅ Downloaded via CLI.")
        except Exception as e2:
            print(f"   ⚠️ CLI failed: {e2}")
else:
    download_success = True
    print("   ✅ Dataset already exists.")

if not download_success:
    raise RuntimeError("❌ Could not download dataset. Check your token and try again.")

# ─── Show what we got ───
print(f"\n📁 Dataset location: {DATASET_DIR}")
all_files = []
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))
exts = {}
for f in all_files:
    ext = os.path.splitext(f)[1].lower()
    exts[ext] = exts.get(ext, 0) + 1
print(f"   📊 Total files: {len(all_files)}")
print(f"   📊 By type: {exts}")

# ─── Find database ───
for f in all_files:
    if f.endswith('.db'):
        DB_PATH = f
        break

# ─── Load data ───
X_data = []
y_data = []

if DB_PATH:
    print(f"\n✅ Found database: {os.path.basename(DB_PATH)}")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [t[0] for t in cursor.fetchall()]
    print(f"   📊 Tables: {tables}")

    # Prefer poseDataset (body + hands)
    target_table = 'poseDataset' if 'poseDataset' in tables else tables[0]
    print(f"   📊 Using: {target_table}")

    cursor.execute(f"PRAGMA table_info({target_table})")
    col_names = [c[1] for c in cursor.fetchall()]
    print(f"   📊 Columns ({len(col_names)}): first={col_names[:3]}, last={col_names[-3:]}")

    cursor.execute(f"SELECT * FROM {target_table}")
    rows = cursor.fetchall()
    print(f"   📊 Rows: {len(rows)}")

    label_col_idx = col_names.index('label') if 'label' in col_names else len(col_names) - 1
    skip_cols = {'id', 'index', 'label'}
    feature_indices = [i for i in range(len(col_names)) if col_names[i] not in skip_cols]

    for row in rows:
        try:
            features = [float(row[i]) if row[i] is not None else 0.0 for i in feature_indices]
            label = str(row[label_col_idx]).strip()
            if label:
                X_data.append(features)
                y_data.append(label)
        except:
            pass

    conn.close()
else:
    # Load from JSON files
    print("\n⏳ Loading from JSON files...")
    json_files = [f for f in all_files if f.endswith('.json')]
    print(f"   📊 Found {len(json_files)} JSON files")
    for jpath in json_files:
        label = os.path.basename(os.path.dirname(jpath))
        try:
            with open(jpath, 'r') as jf:
                data = json.load(jf)
            if 'people' in data and len(data['people']) > 0:
                person = data['people'][0]
                features = []
                for key in ['pose_keypoints_2d', 'hand_left_keypoints_2d', 'hand_right_keypoints_2d']:
                    if key in person:
                        features.extend([float(v) for v in person[key]])
                if len(features) > 0:
                    X_data.append(features)
                    y_data.append(label)
        except:
            pass
    print(f"   📊 Loaded {len(X_data)} samples from JSON")

if len(X_data) == 0:
    raise RuntimeError("❌ No data loaded! The dataset might have a different structure. Check files above.")

X = np.array(X_data)
y = np.array(y_data)
print(f"\n📊 Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"📊 Labels ({len(np.unique(y))}): {list(np.unique(y))}")

# ─── Train ───
print("\n⏳ Training classifier...")
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
clf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"   ✅ Accuracy: {acc*100:.1f}%")
print(f"   ✅ Features: {clf.n_features_in_}")
print(f"   ✅ Classes ({len(le.classes_)}): {list(le.classes_)}")

MODEL_PATH = "/content/psl_classifier_new.pkl"
with open(MODEL_PATH, 'wb') as f:
    pickle.dump((clf, le), f)
print(f"\n💾 Saved: {MODEL_PATH}")
print("🎉 Proceed to Cell 3!")

In [ ]:
#@title 🧠 Cell 3: Setup Live Engine
#@markdown MediaPipe Holistic extracts body + hands for live inference.

import cv2, pickle, logging
import numpy as np
import mediapipe as mp
from collections import deque

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("wasel")

MODEL_PATH = "/content/psl_classifier_new.pkl"
with open(MODEL_PATH, 'rb') as f:
    classifier, label_encoder = pickle.load(f)
N_FEAT = classifier.n_features_in_
print(f"   ✅ Classifier: {N_FEAT} features, {len(label_encoder.classes_)} classes")
print(f"   🏷️ {list(label_encoder.classes_)}")

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False, model_complexity=1,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)
print("   ✅ MediaPipe Holistic loaded.")

def extract_features(results):
    features = []
    # Body pose (x,y)
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 66)
    # Left hand (x,y)
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 42)
    # Right hand (x,y)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 42)

    feat = np.array(features)
    if len(feat) > N_FEAT:
        feat = feat[:N_FEAT]
    elif len(feat) < N_FEAT:
        feat = np.pad(feat, (0, N_FEAT - len(feat)))
    return feat

buf = deque(maxlen=10)

def process_frame(image):
    if image is None:
        return image
    try:
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        out = image.copy()
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

        label_text = "Waiting for signs..."
        feat = extract_features(results)
        buf.append(feat)
        if len(buf) >= 3:
            avg = np.mean(np.array(list(buf)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 25.0:
                label = label_encoder.inverse_transform([idx])[0]
                label_text = f"{label} ({conf:.1f}%)"

        cv2.putText(out, label_text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        return out
    except Exception as e:
        logger.error(f"Frame error: {e}")
        return image

print("\n🚀 Engine ready! Proceed to Cell 4.")

In [ ]:
#@title 🎥 Cell 4: Launch Live Translator

import gradio as gr

demo = gr.Interface(
    fn=process_frame,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Camera"),
    outputs=gr.Image(label="🤟 Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — PSL Translator",
    description="Show PSL signs — AI detects body + hands + translates!",
)
print("🎉 Launching...")
demo.launch(share=True, quiet=False)